# 🧠 Train COOLANT Model for Vietnamese Misinformation Detection

This notebook implements the training pipeline for the COOLANT model, adapted for Vietnamese news data using ResNet50 image features and BERT text features.


In [1]:
import sys
import os
from pathlib import Path

# Detect if running on Google Colab
try:
    from google.colab import drive

    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    drive.mount("/content/drive")
    BASE_PATH = "/content/drive/MyDrive/Thesis_Final/fake-new-detection"
else:
    # Local path - project root is one level up from notebooks directory
    cwd = Path.cwd()
    if cwd.name == "notebooks":
        BASE_PATH = str(cwd.parent)
    else:
        BASE_PATH = str(cwd)

print(f"Using BASE_PATH: {BASE_PATH}")

# Add to sys.path BEFORE any src imports
if BASE_PATH not in sys.path:
    sys.path.insert(0, BASE_PATH)
    print(f"Added {BASE_PATH} to sys.path")

# Verify src module is accessible
try:
    import src

    print(f"✓ src module found at: {src.__file__}")
except ImportError as e:
    print(f"⚠ src module not found: {e}")
    # List what's in BASE_PATH for debugging
    print(f"Contents of {BASE_PATH}:")
    for item in os.listdir(BASE_PATH)[:10]:
        print(f"  - {item}")

# Now safe to import src modules
from src.processing.hdf5_dataset import create_hdf5_dataloaders
import src.models.coolant_official as _co_mod
from src.models.base import FastCNN

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import random
import json
import warnings
import datetime
import h5py
import mlflow
import mlflow.pytorch
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

warnings.filterwarnings("ignore")
print("\nAll imports successful! ✓")

Using BASE_PATH: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-new-detection
Added /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-new-detection to sys.path
✓ src module found at: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-new-detection/src/__init__.py


/opt/miniconda3/envs/fake_news/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



All imports successful! ✓


In [2]:
## 1. Setup & Environment

# Device setup
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Device: {DEVICE}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Paths
HDF5_PATH = os.path.join(BASE_PATH, "data/processed_data/crawled/dataset.h5")
# Fallback check if data directory is different
if not os.path.exists(HDF5_PATH):
    HDF5_PATH = os.path.join(BASE_PATH, "notebooks/processed_data/crawled/dataset.h5")

print(f"HDF5 Data Path: {HDF5_PATH}")

Device: mps
HDF5 Data Path: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-new-detection/notebooks/processed_data/crawled/dataset.h5


In [3]:
## 2. Load Preprocessed Data (HDF5)

BATCH_SIZE = 32

if os.path.exists(HDF5_PATH):
    train_loader, val_loader, test_loader = create_hdf5_dataloaders(
        HDF5_PATH, batch_size=BATCH_SIZE, num_workers=0
    )
    print(f"\nDataLoaders ready ✓")
    print(f"  Train: {len(train_loader.dataset)} samples")
    print(f"  Val:   {len(val_loader.dataset)} samples")
    print(f"  Test:  {len(test_loader.dataset)} samples")
else:
    print(f"❌ HDF5 file not found at {HDF5_PATH}. Please run conversion script first.")

HDF5Dataset [train]: 3369 samples
HDF5Dataset [val]: 722 samples
HDF5Dataset [test]: 723 samples

DataLoaders created:
  Train: 106 batches
  Val:   23 batches
  Test:  23 batches

DataLoaders ready ✓
  Train: 3369 samples
  Val:   722 samples
  Test:  723 samples


In [4]:
## 3. Model Initialization (ResNet50-compatible COOLANT)

IMAGE_DIM = 2048  # ResNet50 output dim
TEXT_EMBED = 768  # BERT hidden size

CONFIG = {
    "shared_dim": 128,
    "sim_dim": 64,
    "clip_embed_dim": 64,
    "feature_dim": 96,  # 16+16+64
    "h_dim": 64,
    "lr": 1e-3,
    "l2": 0.0,
    "num_epochs": 30,
    "seed": SEED,
    "device": str(DEVICE),
    "save_dir": os.path.join(BASE_PATH, "training/checkpoints"),
}


class ResNetCOOLANT(_co_mod.COOLANT_Official):
    """COOLANT_Official concrete wrapper for ResNet50 2048-dim image features."""

    def encode_text(self, text):
        dummy_img = torch.zeros(text.size(0), IMAGE_DIM, device=text.device)
        t, _ = self.similarity_module.encoding(text, dummy_img)
        return t

    def encode_image(self, image):
        dummy_txt = torch.zeros(image.size(0), 30, TEXT_EMBED, device=image.device)
        _, i = self.similarity_module.encoding(dummy_txt, image)
        return i

    def fuse_modalities(self, text_f, image_f):
        return torch.cat([text_f, image_f], dim=-1)


model = ResNetCOOLANT(CONFIG)


# ── Patching internal layers for ResNet50 compatibility ──────────────────────
def _patch_enc(enc):
    """Patch EncodingPart.shared_image from 512 → IMAGE_DIM"""
    layers, done = [], False
    for l in enc.shared_image:
        if isinstance(l, nn.Linear) and not done:
            layers.append(nn.Linear(IMAGE_DIM, l.out_features))
            done = True
        else:
            layers.append(l)
    enc.shared_image = nn.Sequential(*layers)


_patch_enc(model.similarity_module.encoding)
_patch_enc(model.detection_module.encoding)
_patch_enc(model.detection_module.ambiguity_module.encoding)


# ── Patch CLIP projections ──────────────────────────────────────────────────
def _patch_clip_projection(clip_module, is_image=True):
    """Patch CLIP text or image projection to correct input dim"""
    target_dim = IMAGE_DIM if is_image else TEXT_EMBED
    proj = clip_module.image_projection if is_image else clip_module.text_projection

    layers, done = [], False
    for l in proj:
        if isinstance(l, nn.Linear) and not done:
            layers.append(nn.Linear(target_dim, l.out_features))
            done = True
        else:
            layers.append(l)

    new_proj = nn.Sequential(*layers)
    if is_image:
        clip_module.image_projection = new_proj
    else:
        clip_module.text_projection = new_proj


# Patch both image and text projections
_patch_clip_projection(model.clip_module, is_image=True)
_patch_clip_projection(model.clip_module, is_image=False)


# ── Patch FastCNN input dimension ────────────────────────────────────────────
def _patch_cnn(m, input_dim):
    """Recreate FastCNN with correct input dimension"""
    channel = 32
    kernel_size = (1, 2, 4, 8)

    new_cnn = nn.ModuleList()
    for kernel in kernel_size:
        new_cnn.append(
            nn.Sequential(
                nn.Conv1d(input_dim, channel, kernel_size=kernel),
                nn.BatchNorm1d(channel),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.AdaptiveMaxPool1d(1),
            )
        )
    m.fast_cnn = new_cnn


# Apply patches to all FastCNN modules
_patch_cnn(model.similarity_module.encoding.shared_text_encoding, TEXT_EMBED)
_patch_cnn(model.detection_module.encoding.shared_text_encoding, TEXT_EMBED)
_patch_cnn(
    model.detection_module.ambiguity_module.encoding.shared_text_encoding, TEXT_EMBED
)

model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
print("Model ready ✓")

Trainable parameters: 5,610,288
Model ready ✓


In [5]:
## 4. Training Configuration & Helpers

optim_sim = torch.optim.Adam(
    model.similarity_module.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["l2"]
)
optim_clip = torch.optim.AdamW(
    model.clip_module.parameters(), lr=1e-3, weight_decay=5e-4
)
optim_det = torch.optim.Adam(
    model.detection_module.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["l2"]
)

sch_sim = torch.optim.lr_scheduler.StepLR(optim_sim, step_size=10, gamma=0.5)
sch_clip = torch.optim.lr_scheduler.StepLR(optim_clip, step_size=10, gamma=0.5)
sch_det = torch.optim.lr_scheduler.StepLR(optim_det, step_size=10, gamma=0.5)

loss_cos = nn.CosineEmbeddingLoss(margin=0.2)
loss_ce = nn.CrossEntropyLoss()
loss_kl = nn.KLDivLoss(reduction="batchmean")

NUM_EPOCHS = CONFIG["num_epochs"]
SAVE_DIR = Path(CONFIG["save_dir"])
SAVE_DIR.mkdir(parents=True, exist_ok=True)


def make_sim_pairs(text, image, label):
    """Paired/unpaired samples from real-news rows for similarity learning."""
    idx = (label == 0).nonzero(as_tuple=True)[0].tolist() or list(range(2))
    t = text[idx].clone()
    i_match = image[idx].clone()
    i_unmatch = image[idx].clone().roll(3, 0)
    return t, i_match, i_unmatch


def soft_xe(logits, soft_target):
    return -(soft_target * F.log_softmax(logits, 1)).sum() / logits.size(0)


def run_epoch(loader, train=True, epoch_num=None):
    if train:
        model.train()
    else:
        model.eval()

    tot_loss, tot_ok, tot_n = 0.0, 0, 0
    all_y, all_p = [], []
    epoch_sim_loss, epoch_clip_loss, epoch_det_loss = 0.0, 0.0, 0.0
    num_batches = 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for text, image, label in tqdm(
            loader, desc="Training" if train else "Evaluating"
        ):
            text = text.to(DEVICE)
            image = image.to(DEVICE)
            label = label.to(DEVICE)
            bs = label.size(0)

            # ── Task 1a: Cosine Similarity ────────────────────────────────
            ft, fi_m, fi_u = make_sim_pairs(text, image, label)
            ta_m, ia_m, _ = model.similarity_module(ft, fi_m)
            ta_u, ia_u, _ = model.similarity_module(ft, fi_u)
            t_cat = torch.cat([ta_m, ta_u])
            i_cat = torch.cat([ia_m, ia_u])
            y_cos = torch.cat(
                [
                    torch.ones(ta_m.size(0), device=DEVICE),
                    -torch.ones(ta_u.size(0), device=DEVICE),
                ]
            )
            ls = loss_cos(t_cat, i_cat, y_cos)
            epoch_sim_loss += ls.item()
            if train:
                optim_sim.zero_grad()
                ls.backward()
                optim_sim.step()

            # ── Task 1b: CLIP Contrastive ─────────────────────────────────
            # If text is [B, 30, 768], pool to [B, 768]
            text_pooled = text.mean(1) if text.dim() == 3 else text
            ie, te = model.clip_module(image, text_pooled)
            logits = ie @ te.T * math.exp(0.07)
            ids = torch.arange(bs, device=DEVICE)

            # Get soft targets from SimilarityModule
            ts, is_, _ = model.similarity_module(text, image)
            soft_m = is_ @ ts.T * math.exp(0.07)

            lc = (loss_ce(logits, ids) + loss_ce(logits.T, ids)) / 2
            ls2 = (
                soft_xe(logits, F.softmax(soft_m, 1))
                + soft_xe(logits.T, F.softmax(soft_m.T, 1))
            ) / 2
            l_clip = lc + 0.2 * ls2
            epoch_clip_loss += l_clip.item()
            if train:
                optim_clip.zero_grad()
                l_clip.backward()
                optim_clip.step()

            # ── Task 2: Detection ─────────────────────────────────────────
            with torch.no_grad() if not train else torch.enable_grad():
                # Re-encode for detection using latest CLIP weights
                ie2, te2 = model.clip_module(image, text_pooled)
            det, attn, skl = model.detection_module(text, image, te2, ie2)
            ld = loss_ce(det, label) + 0.5 * loss_kl(
                F.log_softmax(attn, 1), F.softmax(skl, 1)
            )
            epoch_det_loss += ld.item()
            if train:
                optim_det.zero_grad()
                ld.backward()
                optim_det.step()

            # Metrics
            pred = det.argmax(1)
            tot_ok += pred.eq(label).sum().item()
            tot_n += bs
            tot_loss += ld.item() * bs
            all_y.extend(label.cpu().numpy())
            all_p.extend(pred.cpu().numpy())
            num_batches += 1

    if epoch_num is not None:
        prefix = "train" if train else "val"
        mlflow.log_metrics(
            {
                f"{prefix}_sim_loss": epoch_sim_loss / num_batches,
                f"{prefix}_clip_loss": epoch_clip_loss / num_batches,
                f"{prefix}_det_loss": epoch_det_loss / num_batches,
                f"{prefix}_acc": tot_ok / tot_n,
            },
            step=epoch_num,
        )

    return tot_loss / tot_n, tot_ok / tot_n, all_y, all_p


print("Training configuration ready ✓")

Training configuration ready ✓


In [6]:
## 5. MLflow Experiment Setup
MLFLOW_EXPERIMENT_NAME = "coolant-vietnamese-misinformation"
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
mlflow_run_name = f"resnet50-coolant-vn-{timestamp}"
mlflow.start_run(run_name=mlflow_run_name)

mlflow.log_params(CONFIG)
print(f"MLflow run started: {mlflow_run_name}")

MLflow run started: resnet50-coolant-vn-20260406_092641


In [7]:
## 6. Training Loop
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

print(f"Starting training for {NUM_EPOCHS} epochs...")

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc, _, _ = run_epoch(train_loader, train=True, epoch_num=epoch)
    vl_loss, vl_acc, vl_y, vl_p = run_epoch(val_loader, train=False, epoch_num=epoch)

    sch_sim.step()
    sch_clip.step()
    sch_det.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(vl_loss)
    history["val_acc"].append(vl_acc)

    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS}]  "
        f"train loss={tr_loss:.4f} acc={tr_acc:.4f}  |  "
        f"val loss={vl_loss:.4f} acc={vl_acc:.4f}"
    )

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), SAVE_DIR / "best_model.pth")
        mlflow.log_artifact(
            str(SAVE_DIR / "best_model.pth"), artifact_path="checkpoints"
        )
        print(f"  ✓ New best val_acc={best_val_acc:.4f} saved.")

print(f"\nTraining complete. Best val acc: {best_val_acc:.4f}")

Starting training for 30 epochs...


Evaluating: 100%|██████████| 23/23 [04:54<00:00, 12.82s/it]


Epoch [01/30]  train loss=0.6409 acc=0.6631  |  val loss=0.6243 acc=0.7022
  ✓ New best val_acc=0.7022 saved.


Evaluating: 100%|██████████| 23/23 [04:48<00:00, 12.55s/it]


Epoch [02/30]  train loss=0.6231 acc=0.6958  |  val loss=0.6164 acc=0.6994


...

Epoch [07/30]  train loss=0.5907 acc=0.6978  |  val loss=0.6217 acc=0.7008


Evaluating: 100%|██████████| 23/23 [04:42<00:00, 12.27s/it]


Epoch [08/30]  train loss=0.5758 acc=0.7038  |  val loss=0.6340 acc=0.7022


Training:   7%|▋         | 7/106 [01:40<23:39, 14.34s/it]


KeyboardInterrupt: 

In [ ]:
## 7. Final Evaluation
model.load_state_dict(torch.load(SAVE_DIR / "best_model.pth", map_location=DEVICE))
te_loss, te_acc, te_y, te_p = run_epoch(test_loader, train=False)

report = classification_report(te_y, te_p, target_names=["Real", "Fake"])
print(f"\n=== Test Results ===")
print(f"Accuracy: {te_acc:.4f}")
print("\nClassification Report:")
print(report)

with open(SAVE_DIR / "test_report.txt", "w") as f:
    f.write(report)
mlflow.log_artifact(str(SAVE_DIR / "test_report.txt"), artifact_path="results")

mlflow.end_run()